# Phase 2 — EDA & Demand Feature Engineering

In [ ]:
# ================================
# PHASE 2: EDA & FEATURE ENGINEERING
# IRS-1: AI-Driven Demand Intelligence
# ================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Always use absolute path — same as Phase 1
BASE_PATH = 'C:/Users/Dell/Desktop/Naveed/gitdemo/Forecasting-future'

print("✅ Libraries loaded")
print("BASE_PATH:", BASE_PATH)

In [ ]:
# ── Load Phase 1 saved files ──────────────────────────────────────
kaggle = pd.read_csv(f'{BASE_PATH}/data/processed/kaggle_unified.csv')
print(f"✅ Kaggle loaded:   {kaggle.shape}")
print(f"   Columns: {kaggle.columns.tolist()[:8]}...")

fresh = pd.read_csv(
    f'{BASE_PATH}/data/processed/fresh_processed.csv',
    usecols=['store_id', 'product_id', 'dt',
             'sale_amount', 'discount', 'holiday_flag',
             'avg_temperature', 'avg_humidity',
             'year', 'month', 'day_of_week', 'is_weekend']
)
print(f"✅ FreshRetailNet:  {fresh.shape}")

print("\n🎉 Both datasets ready for Phase 2!")

In [ ]:
# ── Use Kaggle demand data as primary forecasting dataset ─────────
demand = kaggle.copy()

# Fix date column
demand['Date'] = pd.to_datetime(demand['Date'])

# Rename for consistency
demand = demand.rename(columns={
    'Date':           'date',
    'Store ID_x':     'store_id',
    'Product ID':     'product_id',
    'Sales Quantity': 'sales',
    'Price_x':        'price',
    'Promotions':     'promo'
})

# Fix types
demand['sales'] = pd.to_numeric(demand['sales'], errors='coerce')
demand['promo'] = demand['promo'].map({'Yes': 1, 'No': 0})
demand = demand.sort_values(['store_id', 'product_id', 'date'])
demand = demand.reset_index(drop=True)

print("✅ Demand dataset ready")
print("Shape:", demand.shape)
print("\nDate range:", demand['date'].min(), "→", demand['date'].max())
print("Unique stores:", demand['store_id'].nunique())
print("Unique products:", demand['product_id'].nunique())
print("\nSample:")
print(demand[['date','store_id','product_id','sales','price','promo']].head())

In [ ]:
# ── EDA 1: Sales Distribution ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Sales Distribution Analysis', fontsize=14, fontweight='bold')

# Histogram
axes[0].hist(demand['sales'].dropna(), bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Sales Quantity Distribution')
axes[0].set_xlabel('Sales Quantity')
axes[0].set_ylabel('Frequency')

# Boxplot
axes[1].boxplot(demand['sales'].dropna(), patch_artist=True,
                boxprops=dict(facecolor='lightblue'))
axes[1].set_title('Sales Boxplot')
axes[1].set_ylabel('Sales Quantity')

# Sales by Promo
promo_sales = demand.groupby('promo')['sales'].mean()
axes[2].bar(['No Promo (0)', 'Promo (1)'],
            promo_sales.values,
            color=['#E74C3C', '#27AE60'])
axes[2].set_title('Average Sales: Promo vs No Promo')
axes[2].set_ylabel('Average Sales')

plt.tight_layout()
plt.savefig(f'{BASE_PATH}/data/processed/eda_sales_distribution.png', dpi=150)
plt.show()

print("\nSales Statistics:")
print(demand['sales'].describe())

In [ ]:
# ── EDA 2: Sales Trend Over Time ──────────────────────────────────
daily_sales = demand.groupby('date')['sales'].sum().reset_index()

fig, axes = plt.subplots(2, 1, figsize=(16, 8))
fig.suptitle('Sales Trends Over Time', fontsize=14, fontweight='bold')

# Daily sales
axes[0].plot(daily_sales['date'], daily_sales['sales'],
             color='steelblue', linewidth=0.8, alpha=0.8)
axes[0].set_title('Daily Total Sales')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Total Sales')
axes[0].grid(True, alpha=0.3)

# Monthly average
demand['month_year'] = demand['date'].dt.to_period('M')
monthly = demand.groupby('month_year')['sales'].mean()
axes[1].bar(range(len(monthly)), monthly.values, color='teal', alpha=0.8)
axes[1].set_title('Monthly Average Sales')
axes[1].set_xticks(range(len(monthly)))
axes[1].set_xticklabels([str(p) for p in monthly.index], rotation=45)
axes[1].set_ylabel('Average Sales')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'{BASE_PATH}/data/processed/eda_sales_trend.png', dpi=150)
plt.show()

In [ ]:
# ── EDA 3: Store and Product Analysis ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Store and Product Sales Analysis', fontsize=14, fontweight='bold')

# Top 15 stores by sales
store_sales = demand.groupby('store_id')['sales'].mean().sort_values(ascending=False).head(15)
axes[0].barh(range(len(store_sales)), store_sales.values, color='steelblue')
axes[0].set_yticks(range(len(store_sales)))
axes[0].set_yticklabels([f'Store {s}' for s in store_sales.index])
axes[0].set_title('Top 15 Stores by Average Sales')
axes[0].set_xlabel('Average Sales')

# Top 15 products by sales
product_sales = demand.groupby('product_id')['sales'].mean().sort_values(ascending=False).head(15)
axes[1].barh(range(len(product_sales)), product_sales.values, color='teal')
axes[1].set_yticks(range(len(product_sales)))
axes[1].set_yticklabels([f'P-{p}' for p in product_sales.index])
axes[1].set_title('Top 15 Products by Average Sales')
axes[1].set_xlabel('Average Sales')

plt.tight_layout()
plt.savefig(f'{BASE_PATH}/data/processed/eda_store_product.png', dpi=150)
plt.show()

In [ ]:
# ── ABC-XYZ Product Classification ───────────────────────────────
print("Running ABC-XYZ Classification...")

# ABC — classify by total revenue
product_revenue = demand.groupby('product_id')['sales'].sum().sort_values(ascending=False)
cumulative_pct   = product_revenue.cumsum() / product_revenue.sum()

def abc_class(pct):
    if pct <= 0.70: return 'A'
    elif pct <= 0.90: return 'B'
    else: return 'C'

abc = cumulative_pct.apply(abc_class).rename('abc_class')

# XYZ — classify by demand variability
mean_sales = demand.groupby('product_id')['sales'].mean()
std_sales  = demand.groupby('product_id')['sales'].std()
cv         = (std_sales / mean_sales).rename('cv')

def xyz_class(val):
    if val < 0.5:  return 'X'   # stable
    elif val < 1.0: return 'Y'  # moderate
    else:           return 'Z'  # volatile

xyz = cv.apply(xyz_class).rename('xyz_class')

# Combine
segments = pd.concat([abc, xyz, cv], axis=1)
segments['segment'] = segments['abc_class'] + segments['xyz_class']
segments = segments.reset_index()

print("✅ ABC-XYZ Classification Done!")
print("\nSegment Distribution:")
print(segments['segment'].value_counts().sort_index())

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
segments['abc_class'].value_counts().plot(kind='bar', ax=axes[0],
    color=['#27AE60','#F39C12','#E74C3C'], edgecolor='white')
axes[0].set_title('ABC Classification (by Revenue)')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Number of Products')

segments['xyz_class'].value_counts().plot(kind='bar', ax=axes[1],
    color=['#3498DB','#9B59B6','#E67E22'], edgecolor='white')
axes[1].set_title('XYZ Classification (by Variability)')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Number of Products')

plt.tight_layout()
plt.savefig(f'{BASE_PATH}/data/processed/eda_abc_xyz.png', dpi=150)
plt.show()

In [ ]:
# ── Feature Engineering ───────────────────────────────────────────
print("Building features...")

df = demand.copy()
df = df.sort_values(['store_id', 'product_id', 'date'])
df = df.reset_index(drop=True)

# ── Time features ─────────────────────────────────────────────────
df['day_of_week']  = df['date'].dt.dayofweek
df['month']        = df['date'].dt.month
df['quarter']      = df['date'].dt.quarter
df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)
df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)
df['year']         = df['date'].dt.year

# ── Lag features — max 3 (suits 2.4 avg rows per product) ────────
for lag in [1, 2, 3]:
    df[f'sales_lag_{lag}'] = df.groupby(
        ['store_id', 'product_id'])['sales'].shift(lag)

# ── Rolling features — small windows with min_periods=1 ──────────
for window in [2, 3]:
    df[f'rolling_mean_{window}'] = df.groupby(
        ['store_id', 'product_id'])['sales'].transform(
        lambda x: x.shift(1).rolling(
            window, min_periods=1).mean())
    df[f'rolling_std_{window}'] = df.groupby(
        ['store_id', 'product_id'])['sales'].transform(
        lambda x: x.shift(1).rolling(
            window, min_periods=1).std().fillna(0))

# ── Fourier seasonality features ──────────────────────────────────
t = (df['date'] - df['date'].min()).dt.days
for k in range(1, 4):
    df[f'sin_{k}'] = np.sin(2 * np.pi * k * t / 365.25)
    df[f'cos_{k}'] = np.cos(2 * np.pi * k * t / 365.25)

# ── Price rolling mean ─────────────────────────────────────────────
df['price_rolling_mean'] = df.groupby(
    ['store_id', 'product_id'])['price'].transform(
    lambda x: x.shift(1).rolling(2, min_periods=1).mean())

print("✅ Features created!")
print("Shape:", df.shape)
print("\nNaN counts in lag columns:")
for col in ['sales_lag_1','sales_lag_2','sales_lag_3']:
    print(f"  {col}: {df[col].isnull().sum()} NaN rows")
print(f"\nProducts with lag_1 available: "
      f"{df['sales_lag_1'].notna().sum()} rows")

In [ ]:
# ── Merge ABC-XYZ segments ────────────────────────────────────────
df = df.merge(
    segments[['product_id','segment','abc_class','xyz_class','cv']],
    on='product_id', how='left'
)

# ── Drop NaN only from lag_1 (only first row per product lost) ────
df_features = df.dropna(subset=['sales_lag_1']).reset_index(drop=True)

print("Shape before drop:", df.shape)
print("Shape after drop: ", df_features.shape)
print(f"Rows kept: {len(df_features):,} out of {len(df):,}")

print("\nSegment distribution:")
print(df_features['segment'].value_counts().sort_index())

print("\nFeature columns created:")
feature_cols = [c for c in df_features.columns if any(
    x in c for x in ['lag','rolling','sin','cos',
                      'day_of_week','month','quarter',
                      'week','weekend','year','price_rolling'])]
print(feature_cols)

# ── Save ──────────────────────────────────────────────────────────
df_features.to_csv(
    f'{BASE_PATH}/data/processed/demand_features.csv',
    index=False
)
segments.to_csv(
    f'{BASE_PATH}/data/processed/product_segments.csv',
    index=False
)

print(f"\n✅ demand_features.csv  → {df_features.shape}")
print(f"✅ product_segments.csv → {segments.shape}")
print("\n🎉 PHASE 2 COMPLETE!")

In [ ]:
# Quick check — how many rows per product?
rows_per_product = demand.groupby('product_id').size()

print("Rows per product:")
print(f"  Minimum : {rows_per_product.min()}")
print(f"  Maximum : {rows_per_product.max()}")
print(f"  Average : {rows_per_product.mean():.1f}")
print(f"  Median  : {rows_per_product.median()}")
print(f"\nProducts with >= 28 rows: {(rows_per_product >= 28).sum()}")
print(f"Products with >= 7 rows:  {(rows_per_product >= 7).sum()}")
print(f"Products with >= 2 rows:  {(rows_per_product >= 2).sum()}")
print(f"Total unique products:    {rows_per_product.count()}")